In [ ]:
# Week 3 - Attention maps and naive segmentation
%pip -q install "datasets>=3,<5" "huggingface_hub>=0.27,<2" "scikit-learn>=1.5,<2" matplotlib pandas pillow tqdm astropy nbformat


In [ ]:
# importing necessary libraries
import os, json, random, gc, hashlib, contextlib, io, shutil
from pathlib import Path

os.environ["XFORMERS_DISABLED"] = "1"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Checking the selected option
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
# Turn on for caches/results to survive Colab runtime resets.
USE_DRIVE = False


In [ ]:
# Checking the selected option
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Week-3")
else:
    ROOT = Path("/content/Week-3")
ROOT.mkdir(parents=True, exist_ok=True)
PREPROCESS = transforms.Compose(
    [
        transforms.Resize(
            (224, 224),
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)


In [ ]:
# Defining amp context
def amp_context():
    return (
        torch.autocast("cuda", dtype=torch.float16)
        if DEVICE.type == "cuda"
        else contextlib.nullcontext()
    )


In [ ]:
# Defining to rgb
def to_rgb(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(io.BytesIO(value["bytes"])).convert("RGB")
        if value.get("path"):
            return Image.open(value["path"]).convert("RGB")
    a = np.asarray(value)
    if a.ndim == 3 and a.shape[0] == 3 and a.shape[-1] != 3:
        a = a.transpose(1, 2, 0)
    if a.dtype != np.uint8:
        a = np.nan_to_num(a)
        if a.max() <= 1:
            a = a * 255
        a = np.clip(a, 0, 255).astype(np.uint8)
    return Image.fromarray(a).convert("RGB")


In [ ]:
# Choose input cutouts
SOURCE = "gz10"  # 'gz10', 'grz_npy', or 'rgb_folder'
GRZ_PATH = "/content/desi_grz.npy"
RGB_FOLDER = "/content/galaxies"
N_IMAGES = 10
# Optional explicit indices after inspecting contact sheet; include isolated/blended/star/extended cases.
SELECTED_INDICES = None
OUT = ROOT / "results" / "attention_maps"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# importing necessary libraries
from astropy.visualization import make_lupton_rgb


def grz_to_rgb(grz):
    a = np.asarray(grz, dtype=np.float32)
    if a.shape[0] != 3 and a.shape[-1] == 3:
        a = a.transpose(2, 0, 1)
    assert a.shape[0] == 3, "Expected channels g,r,z"
    g, r, z = np.nan_to_num(a)
    # Robust display stretch, not photometric calibration; use the Week 1 function for exact matching.
    v = np.stack([g, r, z])
    background = np.percentile(v, 10, axis=(1, 2))
    v = v - background[:, None, None]
    scale = max(float(np.percentile(np.clip(v, 0, None), 99.5)), 1e-6)
    g, r, z = v / scale
    return Image.fromarray(make_lupton_rgb(z, r, g, stretch=0.5, Q=8)).convert("RGB")


In [ ]:
# Checking the selected option
if SOURCE == "gz10":
    from datasets import load_dataset

    raw = load_dataset("MultimodalUniverse/gz10", split="train", streaming=True)
    sample = list(raw.take(100))
    images = [to_rgb(r["rgb_image"]) for r in sample]
    image_ids = [str(r.get("object_id", i)) for i, r in enumerate(sample)]
elif SOURCE == "grz_npy":
    arr = np.load(GRZ_PATH, mmap_mode="r")
    images = [grz_to_rgb(x) for x in arr[:100]]
    image_ids = [f"grz_{i}" for i in range(len(images))]
elif SOURCE == "rgb_folder":
    paths = sorted(
        p
        for p in Path(RGB_FOLDER).glob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )[:100]
    images = [Image.open(p).convert("RGB") for p in paths]
    image_ids = [p.stem for p in paths]
else:
    raise ValueError(SOURCE)


In [ ]:
# Processing the data
assert len(images) > 0, "No images found. Upload your data or use gz10."
fig, axes = plt.subplots(
    (len(images) + 9) // 10,
    10,
    figsize=(16, 1.7 * ((len(images) + 9) // 10)),
    squeeze=False,
)


In [ ]:
# Running the next step
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(str(i), fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Extract attention and threshold 60% of patch attention mass
def mass_mask(attention, mass=0.6):
    """Smallest descending set whose cumulative patch mass reaches the target."""
    if not 0 < mass <= 1:
        raise ValueError("mass must be in (0,1]")
    a = attention.float().clamp_min(0)
    a = a / a.sum(-1, keepdim=True).clamp_min(1e-12)
    values, order = a.sort(dim=-1, descending=True)
    # Include the patch that crosses the target (avoid an off-by-one error).
    keep = (values.cumsum(-1) - values) < mass
    return torch.zeros_like(keep).scatter(-1, order, keep)


In [ ]:
# Defining extract maps
@torch.inference_mode()
def extract_maps(model, image, is_dino=False):
    x = PREPROCESS(image).unsqueeze(0).to(DEVICE)
    if is_dino:
        full = model.get_last_selfattention(x)
        weights = full[0, :, 0, 1:].float().cpu()
    else:
        captured = {}

        def hook(module, args):
            tokens = args[0]
            B, N, C = tokens.shape
            H = module.num_heads
            qkv = module.qkv(tokens).reshape(B, N, 3, H, C // H).permute(2, 0, 3, 1, 4)
            q, k = qkv[0].float(), qkv[1].float()
            # Compute only the CLS query but normalize over ALL keys, including registers.
            a = ((q[:, :, :1] * module.scale) @ k.transpose(-2, -1)).softmax(-1)
            start = 1 + getattr(model, "num_register_tokens", 0)
            captured["weights"] = a[0, :, 0, start:].detach().cpu()

        handle = model.blocks[-1].attn.register_forward_pre_hook(hook)
        try:
            model(x)
        finally:
            handle.remove()
        weights = captured["weights"]
    side = int(weights.shape[-1] ** 0.5)
    assert side * side == weights.shape[-1]
    patch_masks = mass_mask(weights)
    maps = F.interpolate(
        weights.reshape(1, -1, side, side), size=(224, 224), mode="nearest"
    )[0]
    masks = F.interpolate(
        patch_masks.float().reshape(1, -1, side, side), size=(224, 224), mode="nearest"
    )[0].bool()
    return maps.numpy(), masks.numpy(), weights.numpy()


In [ ]:
# Verify threshold semantics on known values.
assert mass_mask(torch.tensor([[0.5, 0.3, 0.2]]), 0.6).tolist() == [[True, True, False]]
assert mass_mask(torch.tensor([[0.6, 0.3, 0.1]]), 0.6).tolist() == [
    [True, False, False]
]


In [ ]:
# Setting indices
indices = (
    SELECTED_INDICES
    if SELECTED_INDICES is not None
    else list(range(min(N_IMAGES, len(images))))
)
assert all(0 <= i < len(images) for i in indices)
models = [
    ("DINO_S8", "facebookresearch/dino:main", "dino_vits8", True),
    ("DINOv2_S14", "facebookresearch/dinov2:main", "dinov2_vits14", False),
    (
        "DINOv2_S14_registers",
        "facebookresearch/dinov2:main",
        "dinov2_vits14_reg",
        False,
    ),
]
summary = []


In [ ]:
# Visualizing the results
for name, repo, entry, is_dino in models:
    model = (
        torch.hub.load(repo, entry, pretrained=True, trust_repo=True).eval().to(DEVICE)
    )
    for i in tqdm(indices, desc=name):
        maps, masks, weights = extract_maps(model, images[i], is_dino)
        fig, axes = plt.subplots(1, 8, figsize=(21, 3))
        rgb = np.asarray(images[i].resize((224, 224)))
        axes[0].imshow(rgb)
        axes[0].set_title(f"RGB: index {i}")
        for h in range(6):
            axes[h + 1].imshow(maps[h], cmap="inferno")
            axes[h + 1].set_title(f"Head {h+1}")
        union = masks.any(0)
        axes[7].imshow(rgb)
        axes[7].imshow(
            np.ma.masked_where(~union, union), alpha=0.4, cmap="spring", vmin=0, vmax=1
        )
        axes[7].set_title("Union: 60% / head")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(name + " — each head independently colour-scaled")
        fig.tight_layout()
        fig.savefig(OUT / f"{name}_{i:03d}.png", dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        np.savez_compressed(
            OUT / f"{name}_{i:03d}.npz",
            attention=maps,
            head_masks=masks,
            union=union,
            patch_attention=weights,
        )
        normalized = weights / weights.sum(-1, keepdims=True)
        summary.append(
            {
                "model": name,
                "index": i,
                "object_id": image_ids[i],
                "union_area_fraction": float(union.mean()),
                "mean_peak_patch_mass": float(normalized.max(-1).mean()),
            }
        )
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


In [ ]:
# Processing the data
pd.DataFrame(summary).to_csv(OUT / "attention_summary.csv", index=False)
display(pd.DataFrame(summary))


In [ ]:
# Interpret, then export
archive = shutil.make_archive(
    "/content/Week3_attention_results", "zip", ROOT / "results" / "attention_maps"
)
from google.colab import files

files.download(archive)
